# iPerturb — reproducibility notebook

Runs the full **iPerturb** pipeline (build context-specific GRN → fit Hill-kinetics GRNN → evaluate) for **K562** and **RPE1**, from a clean clone to a downloadable results zip.

**Before you start:** set a **GPU** runtime (`Runtime → Change runtime type → GPU`).
Expected runtime ~30–60 min (database downloads + training for two cell lines).

You must supply the **GeneHancer v5.26** files yourself (license-gated, not redistributable) — see step 4.

In [ ]:
# 1. Check GPU and clone the repository
!nvidia-smi -L || echo 'No GPU detected — enable one via Runtime > Change runtime type > GPU'
![ -d /content/iPerturb/.git ] || git clone https://github.com/kagtgi/iPerturb.git /content/iPerturb
%cd /content/iPerturb

In [ ]:
# 2. Install dependencies (Colab already has torch/numpy/pandas/etc.; the rest install here)
!pip install -q -r requirements.txt

In [ ]:
# 3. Download the Replogle Perturb-seq expression data into /content
import os, urllib.request
FILES = {'K562.h5ad': '35773219', 'RPE1.h5ad': '35775606'}
for fn, fid in FILES.items():
    dst = f'/content/{fn}'
    if not (os.path.exists(dst) and os.path.getsize(dst) > 1e6):
        print('Downloading', fn, '...')
        urllib.request.urlretrieve(f'https://ndownloader.figshare.com/files/{fid}', dst)
    print(fn, f'{os.path.getsize(dst)/1e6:.0f} MB')

## 4. GeneHancer v5.26 (required, license-gated)

GeneHancer is **not redistributable**, so it cannot ship with the repo. Obtain the three files from GeneHancer / GeneCards and make them available to Colab. Easiest: put them in a Google Drive folder and set `GENEHANCER_DIR` below; the cell mounts Drive and copies them to `/content/`.

Needed: `GeneHancer_v5.26.gff`, `GeneHancer_TFBSs_v5.26.txt`, `GeneHancer_Tissues_v5.26.txt`.

In [ ]:
# 4. Make the GeneHancer files available under /content
GENEHANCER_DIR = '/content/drive/MyDrive/GeneHancer'   # <-- EDIT to your Drive folder

import os, shutil
NEEDED = ['GeneHancer_v5.26.gff', 'GeneHancer_TFBSs_v5.26.txt', 'GeneHancer_Tissues_v5.26.txt']
if not all(os.path.exists(f'/content/{f}') for f in NEEDED):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print('Drive mount skipped:', e)
    for f in NEEDED:
        src = os.path.join(GENEHANCER_DIR, f)
        if os.path.exists(src) and not os.path.exists(f'/content/{f}'):
            print('copying', f, '...'); shutil.copy(src, f'/content/{f}')
# Fallback (slow for ~600 MB): from google.colab import files; files.upload()
missing = [f for f in NEEDED if not os.path.exists(f'/content/{f}')]
assert not missing, f'Missing GeneHancer files: {missing} — put them in {GENEHANCER_DIR} or /content/'
print('GeneHancer files present.')

## 5. Run the full pipeline (K562, then RPE1)
This builds each GRN, trains the GRNN, evaluates on held-out perturbations, and writes figures + metrics under `/content/`. Takes a while on GPU.

In [ ]:
# 5. Run end-to-end
%run /content/iPerturb/iperturb.py

## 6. Collect and download results

In [ ]:
# 6. Zip the outputs (figures + metrics + GRN tables) and download
import os, glob, shutil
os.makedirs('/content/results', exist_ok=True)
if os.path.isdir('/content/grn_plots'):
    shutil.copytree('/content/grn_plots', '/content/results/figures', dirs_exist_ok=True)
PATTERNS = ['/content/*_metrics_subsample_*runs.tsv', '/content/*_grn_logged.tsv',
            '/content/grn_edges.tsv', '/content/grn_full.graphml', '/content/grn_gene_lists.txt']
for pat in PATTERNS:
    for p in glob.glob(pat):
        shutil.copy(p, '/content/results/')
shutil.make_archive('/content/iperturb_results', 'zip', '/content/results')
print('Wrote /content/iperturb_results.zip')
try:
    from google.colab import files
    files.download('/content/iperturb_results.zip')
except Exception as e:
    print('Download manually from the Files panel:', e)